# 第 5 週作業：全自動區域受災衝擊評估系統（動態監測版）
將前兩週的成果（W3 的避難所河川距離、W4 的地形坡度）整合成 ARIA v3.0，並在 **2025 年鳳凰颱風 (Typhoon Fung-wong)** 的極端情境下進行壓力測試。

## 載入使用套件，並讀取避難所(花蓮、宜蘭)資訊
#### 讀取檔案後，確認CSR坐標系統
#### 將避難所資料儲存至變數

In [1]:
# 載入使用套件
import geopandas as gpd
import pandas as pd
import numpy as np
import folium
import requests
import json
import os
from dotenv import load_dotenv
from shapely.geometry import Point

# 1. 載入 .env 環境變數
load_dotenv()

# 2. 讀取避難所資料(json,含河川距離分級），涵蓋地形風險（mean_elevation、max_slope、terrain_risk(risk_level))
file_path = '../data/terrain_risk_audit.json'
with open(file_path, 'r', encoding='utf-8') as f:
    shelter_data = json.load(f)

# 將 JSON 轉為 DataFrame
df_shelters_raw = pd.DataFrame(shelter_data)

# 3. 建立幾何物件 (使用檔案中的 '經度' 與 '緯度')，JSON 格式為 WGS84 (EPSG:4326)
geometry = [Point(xy) for xy in zip(df_shelters_raw['經度'], df_shelters_raw['緯度'])]
gdf_shelters = gpd.GeoDataFrame(df_shelters_raw, geometry=geometry, crs="EPSG:4326")

# 4. 為了後續空間分析與距離計算，轉換至 TWD97 (EPSG:3826, 單位為公尺)
gdf_shelters = gdf_shelters.to_crs("EPSG:3826")

# 5. 欄位對齊 (將中文欄位映射到後續 Cell 預期使用的英文名稱，方便後續程式碼執行)
# column_mapping = {
#     '避難收容處所名稱': 'name',
#     '縣市及鄉鎮市區': 'TOWNNAME',
#     '序號': 'shelter_id'
# }
# gdf_shelters = gdf_shelters.rename(columns=column_mapping)

# 6. 列印資料摘要
print("--- Real Shelter Data Loaded (Hualien) ---")
print(f"Total shelter count: {len(gdf_shelters)}")
print(f"Coordinate Reference System (CRS): {gdf_shelters.crs}")
gdf_shelters.head(2)

--- Real Shelter Data Loaded (Hualien) ---
Total shelter count: 438
Coordinate Reference System (CRS): EPSG:3826


,序號,縣市及鄉鎮市區,村里,避難收容處所地址,經度,緯度,避難收容處所名稱,預計收容村里,預計收容人數,適用災害類別,...,座標有效性,座標問題,座標系統,is_indoor,mean_elevation,std_elevation,max_slope,risk_level,river_distance_category,geometry
0,46,花蓮縣秀林鄉,和平村,和平112號,121.748879,24.308073,和平國小,和中部落,250,"水災,土石流",...,有效,None,EPSG:4326,True,27.502352,17.994240,45.473251,高風險,<1000m,POINT (326011.111 2689346.508)
1,1306,花蓮縣富里鄉,豐南村,豐南7鄰12-3號,121.268600,23.143000,豐南社區活動中心,豐南村,50,"水災,震災,土石流,海嘯",...,有效,None,EPSG:4326,True,299.459141,13.260633,41.046368,極高風險,<500m,POINT (277505.779 2560143.417)


# 抓取雨量資料
### 1. 建立模式切換器 (Mode Switcher)：如果 API 呼叫失敗，則自動切換到最近一次的本地快照(data/scenarios/fungwong_202511.json)
### 2. 建立雨量檔案解析函數，轉換成目標格式

In [2]:
# 確認APP_MODE，若值為LIVE則使用CWA API，若值為SIMULATION則使用本機下載雨量資料
APP_MODE = os.getenv("APP_MODE")
print(APP_MODE)
if APP_MODE == "LIVE":
    print("使用CWA API")
elif APP_MODE == "SIMULATION":
    print("使用本機下載雨量資料")
else:
    print("APP_MODE not set")

SIMULATION
使用本機下載雨量資料


In [3]:
# 抓取即時雨量的FUNCTION
def fetch_cwa_api(api_key):
    """
    呼叫 CWA 降雨 API 並回傳 JSON 資料。
    包含異常處理以確保系統穩定。
    """
    api_key = os.getenv("CWA_API_KEY")
    url = "https://opendata.cwa.gov.tw/api/v1/rest/datastore/O-A0002-001"
    params = {
        "Authorization": api_key,
        "format": "JSON"
    }
    
    try:
        # 1. 發送請求 (設定 timeout 避免程式無限卡死)
        response = requests.get(url, params=params, timeout=10)
        
        # 2. 檢查 HTTP 狀態碼 (200 代表 OK)
        response.raise_for_status()
        
        # 3. 解析 JSON
        data = response.json()
        
        # 4. 驗證 CWA API 內部的成功旗標
        if data.get("success") == "true":
            print(f"✅ CWA API 連線成功！成功取得 {len(data['records']['Station'])} 個測站資料。")
            return data
        else:
            print("❌ API 回傳成功但內容有誤，請檢查 API Key 或參數。")
            return None
            
    except requests.exceptions.RequestException as e:
        # 處理網路連線、Timeout 或 HTTP 錯誤
        print(f"❌ 網路連線發生錯誤: {e}")
        return None
    except Exception as e:
        # 處理其他預期外的錯誤
        print(f"⚠️ 發生未知錯誤: {e}")
        return None


# 解析SIMULATION_DATA格式的函式
def parse_typhoon_json(data):
    stations = data['records']['Station']
    results = []
    
    for s in stations:
        # 提取座標 (處理無標籤結構)
        coords = s['GeoInfo']['Coordinates'][0]
        lat = float(coords['StationLatitude'])
        lon = float(coords['StationLongitude'])
        
        # 提取雨量並改名為 rain_1hr
        # 注意：我們將 Past1hr 的 Precipitation 數值直接當作 rain_1hr
        rain_element = s.get('RainfallElement', {})
        p_1hr = rain_element.get('Past1hr', {}).get('Precipitation', 0.0)
        p_3hr = rain_element.get('Past3hr', {}).get('Precipitation', 0.0)
        p_24hr = rain_element.get('Past24hr', {}).get('Precipitation', 0.0)
        
        results.append({
            'StationName': s['StationName'],
            'StationId': s['StationId'],
            'CountyName': s['GeoInfo']['CountyName'],
            'TownName': s['GeoInfo']['TownName'],
            'lat': lat,
            'lon': lon,
            'rain_1hr': float(p_1hr) if float(p_1hr) > -990 else 0.0,
            'rain_3hr': float(p_3hr) if float(p_3hr) > -990 else 0.0,
            'rain_24hr': float(p_24hr) if float(p_24hr) > -990 else 0.0,
            'geometry': Point(lon, lat)
        })
    
    df = pd.DataFrame(results)
    return gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

# 即時雨量資料解析與 GeoDataFrame 產出
def normalize_cwa_json(raw):
    """偵測 JSON 根目錄格式並回傳測站列表"""
    if not raw:
        return []
    # 你的資料屬於這類：records -> Station
    if "records" in raw and "Station" in raw["records"]:
        return raw["records"]["Station"]
    # 備用格式：cwaopendata -> dataset -> Station
    if "cwaopendata" in raw and "dataset" in raw["cwaopendata"]:
        return raw["cwaopendata"]["dataset"]["Station"]
    return []

# 解析即時雨量檔案回傳格式
def parse_rainfall_json(data):
    """解析嵌套 JSON 並產出 GeoDataFrame (EPSG:4326)"""
    stations = normalize_cwa_json(data)
    results = []
    
    for s in stations:
        # 1. 抓取地理資訊層 (GeoInfo)
        geo_info = s.get("GeoInfo", {})
        
        # 2. 從座標列表中精確選取 WGS84 (避免座標偏移)
        coords = geo_info.get("Coordinates", [])
        lat, lon = None, None
        for c in coords:
            if c.get("CoordinateName") == "WGS84":
                lat = float(c.get("StationLatitude"))
                lon = float(c.get("StationLongitude"))
                break
        
        # 如果沒抓到座標則跳過該站
        if lat is None or lon is None:
            continue
            
        # 3. 處理雨量資料與 -998 無效值
        def get_rain(key):
            rain_el = s.get("RainfallElement", {})
            val = rain_el.get(key, {}).get("Precipitation", "-998")
            try:
                f_val = float(val)
                # 依照要求：將 -998 (無資料) 設為 0.0
                return 0.0 if f_val <= -998 else f_val
            except:
                return 0.0

        # 4. 整合資料欄位
        results.append({
            "StationName": s.get("StationName"),
            "StationId": s.get("StationId"),
            "CountyName": geo_info.get("CountyName"),
            "TownName": geo_info.get("TownName"),
            "lat": lat,
            "lon": lon,
            "rain_1hr": get_rain("Past1hr"),
            "rain_3hr": get_rain("Past3hr"),
            "rain_24hr": get_rain("Past24hr"),
            "geometry": Point(lon, lat)
        })
    
    # 5. 轉為 GeoDataFrame 並設定 CRS
    df = pd.DataFrame(results)
    if df.empty:
        return gpd.GeoDataFrame()
        
    gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")
    return gdf

raw_json=None
# 若APP_MODE為LIVE，則呼叫CWA即時降雨API
if APP_MODE == "LIVE":
    # 嘗試呼叫 CWA 即時降雨 API
    print("📡 正在嘗試呼叫 CWA 即時 API...")
    api_key = os.getenv("CWA_API_KEY")
    raw_json = fetch_cwa_api(api_key)

        # 使用 Cell [3] 定義的統一函式解析 JSON → GeoDataFrame
    if raw_json:
        gdf_rainfall = parse_rainfall_json(raw_json)

    # 檢查回傳結果：如果 API 失敗或為 None，則載入鳳凰颱風模擬資料
    if raw_json is None:
        fallback_path = os.getenv("SIMULATION_DATA")
        print(f"⚠️ API 呼叫失敗或未設定，切換至模擬模式：載入 {fallback_path}")
        
        try:
            with open(fallback_path, 'r', encoding='utf-8') as f:
                raw_json = json.load(f)
                # 執行解析
                gdf_rainfall = parse_typhoon_json(raw_json)
        except FileNotFoundError:
            print(f"❌ 錯誤：找不到路徑下的後備檔案 {fallback_path}")
            raw_json = None


# 若APP_MODE為SIMULATION，則載入本機下載雨量資料
if APP_MODE == "SIMULATION":
    # 使用本機下載雨量資料
    print("🔄 正在載入本機下載雨量資料...")

    fallback_path = os.getenv("SIMULATION_DATA")
    
    try:
        with open(fallback_path, 'r', encoding='utf-8') as f:
            raw_json = json.load(f)
            # 執行解析
            gdf_rainfall = parse_typhoon_json(raw_json)
    except FileNotFoundError:
        print(f"❌ 錯誤：找不到路徑下的後備檔案 {fallback_path}")
        raw_json = None

# 列印實驗要求的資料摘要
print("\n--- Rainfall Data Summary ---")

# 5. 列印處理結果
if gdf_rainfall is not None and not gdf_rainfall.empty:
    print(f"✅ 資料載入成功！共取得 {len(gdf_rainfall)} 個測站。")
    print(f"📍 座標系統檢核：{gdf_rainfall.crs}")
else:
    print("❌ 關鍵錯誤：未成功載入任何降雨資料，後續地圖繪製將受影響。")
gdf_rainfall.head(3)



🔄 正在載入本機下載雨量資料...

--- Rainfall Data Summary ---
✅ 資料載入成功！共取得 1256 個測站。
📍 座標系統檢核：EPSG:4326


,StationName,StationId,CountyName,TownName,lat,lon,rain_1hr,rain_3hr,rain_24hr,geometry
0,國一S072K,CAC010,桃園市,楊梅區,24.895830,121.134200,0.0,0.5,15.0,POINT (121.1342 24.89583)
1,馬光農場,C2K620,雲林縣,虎尾鎮,23.721453,120.378864,0.0,0.0,0.5,POINT (120.37886 23.72145)
2,出雲,C2FB50,臺中市,和平區,24.206469,120.902794,0.0,0.0,1.0,POINT (120.90279 24.20647)


## 建立動態風險

* **座標同步 (CRS Sync)：** 統一轉換至 `EPSG:3826` (TWD97)，確保 **5km 緩衝區** 之物理空間計算精確度。
* **影響圈建模 (Impact Mapping)：** 針對時雨量 > **40mm** 之測站建立 5km 緩衝半徑，定義強降雨輻射區。
* **空間碰撞 (Spatial Nexus)：** * 使用 `Left Join` 確保全台避難所清單完整。
    * 透過 `drop_duplicates` 篩選，強制令單一避難所僅保留其接觸到之 **最大雨量威脅值**。
* **動態風險矩陣 (Risk Logic)：**

| 等級 | 觸發條件 | 建議行動 |
| :--- | :--- | :--- |
| **🔴 CRITICAL** | 雨量 > 80mm | **立即執行** 預防性撤離。 |
| **🟠 URGENT** | 雨量 > 40mm + 地形高風險 | **強制執行** 戒備與物資預部署。 |
| **🟡 WARNING** | 雨量 > 40mm 或 地形高風險 | **加強監測** 降雨路徑與坡度穩定性。 |
| **🟢 SAFE** | 未達上述門檻 | 系統持續掃描，維持常態監控。 |

---

In [4]:
import geopandas as gpd

# 1. 確保投影一致
print("原始 rainfall 投影:", gdf_rainfall.crs)
gdf_rainfall_3826 = gdf_rainfall.to_crs(epsg=3826)
print("轉換後 rainfall 投影:", gdf_rainfall_3826.crs)

# 建立緩衝區，並保留雨量與站名
rain_stations_impact = gdf_rainfall_3826[gdf_rainfall_3826['rain_1hr'] > 0].copy() 
rain_stations_impact['geometry'] = rain_stations_impact.geometry.buffer(5000)

# 3. 空間連接 (使用 left join 以保留所有避難所)
# 這會讓不在 5km 內的避難所，其 rain_1hr 欄位變為 NaN
shelters_analysis = gpd.sjoin(
    gdf_shelters, 
    rain_stations_impact[['rain_1hr', 'StationName', 'geometry']], 
    how='left', 
    predicate='within'
)

# 若一個避難所同時在多個測站範圍內，取雨量最大值，避免重複列
shelters_analysis = shelters_analysis.sort_values('rain_1hr', ascending=False).drop_duplicates(subset='避難收容處所名稱')

# 填補空值（如果 5km 內沒測站，顯示為 "5km內無測站"）
shelters_analysis['StationName'] = shelters_analysis['StationName'].fillna("無鄰近測站")
shelters_analysis['rain_1hr'] = shelters_analysis['rain_1hr'].fillna(0)
# 4. 定義動態風險分級邏輯
def assign_dynamic_risk(row):
    rain = row['rain_1hr'] if not pd.isna(row['rain_1hr']) else 0
    t_risk = str(row['risk_level']) # 假設原來的地形風險欄位是 risk_level
    
    # --- CRITICAL ---
    if rain > 80:
        return '🔴 CRITICAL'
    
    # --- URGENT ---
    elif rain > 40 and ('極高風險' in t_risk or '高風險' in t_risk.upper()):
        return '🟠 URGENT'
    
    # --- WARNING ---
    elif rain > 40 or ('極高風險' in t_risk or '高風險' in t_risk.upper()):
        # 注意：這裡根據你的需求，只要「雨量 > 40」或「地形高風險」任一成立即觸發
        return '🟡 WARNING'
    
    # --- SAFE ---
    else:
        return '🟢 SAFE'

# 5. 執行分級並輸出
shelters_analysis['Dynamic Risk'] = shelters_analysis.apply(assign_dynamic_risk, axis=1)

# 統計各級數量
summary = shelters_analysis['Dynamic Risk'].value_counts()
print("📊 全台避難所動態風險統計：")
print(summary)
shelters_analysis.head(2)
# 顯示高風險清單
# high_risk_list = shelters_analysis[shelters_analysis['dynamic_priority'].str.contains('CRITICAL|URGENT|WARNING')]
# if not high_risk_list.empty:
#     display(high_risk_list[['避難收容處所名稱', 'risk_level', 'rain_1hr', 'dynamic_priority']].head(5))

原始 rainfall 投影: EPSG:4326
轉換後 rainfall 投影: EPSG:3826
📊 全台避難所動態風險統計：
Dynamic Risk
🟡 WARNING     217
🟢 SAFE        111
🔴 CRITICAL     85
🟠 URGENT       24
Name: count, dtype: int64


,序號,縣市及鄉鎮市區,村里,避難收容處所地址,經度,緯度,避難收容處所名稱,預計收容村里,預計收容人數,適用災害類別,...,mean_elevation,std_elevation,max_slope,risk_level,river_distance_category,geometry,index_right,rain_1hr,StationName,Dynamic Risk
231,4011,宜蘭縣蘇澳鎮,聖湖里,中山路二段1號,121.8354,24.5953,蘇澳國中,聖湖里、聖愛里、蘇西里,67,"水災,震災,土石流,海嘯",...,NaN,NaN,NaN,高風險,<500m,POINT (334601.356 2721210.187),250.0,130.5,蘇澳,🔴 CRITICAL
244,4049,宜蘭縣蘇澳鎮,永榮里,大同路108號,121.8349,24.6219,馬賽國小,永榮里、新城里,63,"水災,震災,土石流,海嘯",...,NaN,NaN,NaN,低風險,>1000m,POINT (334532.839 2724156.196),250.0,130.5,蘇澳,🔴 CRITICAL


## 📝 視覺化實作說明 (Technical Implementation)

本模組透過 **Folium** 庫將即時降雨數據轉化為具備空間感知的互動式地圖，主要用於直觀評估花蓮地區的降雨空間分佈強度。

### 1. 地圖基礎設定 (Map Initialization)
* **中心座標：** 鎖定花蓮縣範圍 `[23.98, 121.55]`。
* **縮放層級：** 預設 `zoom_start=10`，兼顧全縣覆蓋與市區細節。
* **底圖選擇：** 使用 `OpenStreetMap` 以確保道路與地形參考資訊完整。

### 2. 動態標記邏輯 (Dynamic Symbology)
為了達到「一眼掌握風險」的效果，系統導入了兩層視覺變量：

* **顏色編碼 (Color Coding)：** 透過 `rain_color` 函式將雨量強度分類：
    * 🟢 **Green:** < 10 mm/hr (安全)
    * 🟡 **Gold:** 10 - 40 mm/hr (注意)
    * 🟠 **Orange:** 40 - 80 mm/hr (警告)
    * 🔴 **Red:** ≥ 80 mm/hr (危險)

* **尺寸縮放 (Radius Scaling)：** 透過 `rain_radius` 函式動態調整圓形標記半徑（`rain_mm / 5`），雨量越大，地圖上的圓圈視覺佔比越顯著。

### 3. 圖層管理協定 (Layer Control)
* **FeatureGroup：** 建立名為 `1小時累積雨量` 的圖層群組。這允許使用者在最終地圖中透過控制面板自由切換該圖層的顯示/隱藏，方便與其他 GIS 圖層（如避難所位置）進行疊圖分析。
* **互動式提示 (Tooltip)：** 在 `CircleMarker` 中整合 HTML 語法，懸停滑鼠即可即時查看「測站名稱」與「確切雨量數值」。

---

In [5]:
import folium
from folium import LayerControl
# 1. & 2. 設定地圖中心點座標為花蓮縣 (緯度 ~23.98, 經度 ~121.55)
hualien_center = [23.98, 121.55]

# 3. & 4. 建立 Folium 地圖物件並賦值給變數 m
# 設定初始縮放等級 zoom_start=10，使用 OpenStreetMap 底圖
m = folium.Map(
    location=hualien_center,
    zoom_start=10,
    tiles='OpenStreetMap'
)

# 1. 定義顏色判斷函式
def rain_color(rain_mm):
    """根據雨量強度回傳對應顏色"""
    if rain_mm < 10:
        return 'green'   # 安全 (Safe)
    elif 10 <= rain_mm < 40:
        return 'gold'    # 注意 (Caution)
    elif 40 <= rain_mm < 80:
        return 'orange'  # 警告 (Warning)
    else:
        return 'red'     # 危險 (Danger)

# 定義半徑縮放函式
def rain_radius(rain_mm):
    """根據雨量強度計算標記半徑，最小為 5"""
    return max(5, rain_mm / 5)
    
# --- 重點：建立一個 FeatureGroup ---
# name 參數會顯示在 LayerControl 的選單中
fg_rain = folium.FeatureGroup(name='1小時累積雨量', overlay=True, control=True)

# 遍歷 gdf_rainfall 並將測站加入「fg_rain」圖層
for idx, row in gdf_rainfall.iterrows():
    rain_val = row['rain_1hr']
    
    # 建立圓形標記並加入到 fg_rain
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=rain_radius(rain_val),
        color='black',
        weight=1,
        fill=True,
        fill_color=rain_color(rain_val),
        fill_opacity=0.7,
        tooltip=f"測站: {row['StationName']}<br>雨量: {rain_val} mm/hr"
    ).add_to(fg_rain) # 注意這裡改成了 add_to(fg_rain)

# 將整個圖層群組加入地圖
fg_rain.add_to(m)

# # 加入圖層控制面板，collapsed=False 代表選單預設展開
# LayerControl(collapsed=False).add_to(m)
# # 顯示地圖預覽
# m


## 📝 視覺化決策介面說明 (Decision Interface Implementation)

本模組將前階段產出的「動態風險分析結果」轉化為地圖上的直觀預警標記，提供給指揮官具備空間背景的決策參考。

### 1. 顏色編碼與視覺心理 (Color Semantics)
系統透過 `risk_color_map` 將抽象的風險等級具象化，使用 Folium 內建圖標顏色進行視覺區隔：
* **🔴 CRITICAL (`darkred`):** 極端威脅，圖標強化視覺衝擊力。
* **🟠 URGENT (`orange`):** 緊急狀態，提醒指揮官優先配置資源。
* **🟡 WARNING (`beige`):** 警戒區域，維持高度關注。
* **🟢 SAFE (`green`):** 目前安全，僅作常態監測。

### 2. 空間投影轉換 (Coordinate Re-projection)
* **關鍵動作：** 在繪製標記前，程式執行 `.to_crs(epsg=4326)`。
* **技術細節：** 由於前階段計算緩衝區 (Buffer) 使用的是 TWD97 (EPSG:3826) 投影（單位為公尺），而 Folium 顯示必須回歸全球地理座標系 (WGS84)，此步驟確保標記精準座落在地圖經緯度上。

### 3. 高互動式資訊視窗 (Interactive HTML Popup)
每個避難所標記均搭載了一個動態生成的 HTML 視窗，整合多維度環境數據：
* **動態風險：** 第一時間顯示系統判定之風險等級。
* **環境數據：** 結合鄰近測站名稱與實測時雨量（若無資料則顯示為 0.0）。
* **背景資訊：** 包含該地點原始地形風險等級、行政區歸屬以及海拔高度，輔助判斷排水或土石流潛勢。

### 4. 圖層管理策略 (Layer Strategy)
* **FeatureGroup：** 將避難所標記打包為「避難所動態風險狀態」獨立圖層。
* **互補性：** 此圖層可與之前的「1小時累積雨量」圖層疊加，實現「降雨來源」與「受災客體」的空間關聯分析。

In [6]:
import folium
from folium import FeatureGroup, LayerControl

# 1. 定義風險等級與顏色對應表 (符合 Folium 支援的 icon 顏色)
risk_color_map = {
    '🔴 CRITICAL': 'darkred',
    '🟠 URGENT': 'orange',
    '🟡 WARNING': 'beige',
    '🟢 SAFE': 'green'
}

# 2. 建立避難所圖層
shelter_layer = FeatureGroup(name="避難所動態風險狀態", show=True)

# 3. 將分析後的結果轉回經緯度 (EPSG:4326) 並遍歷
# 注意：這裡使用的是你分析完後的 shelters_analysis
for idx, row in shelters_analysis.to_crs(epsg=4326).iterrows():
    
    # 取得動態風險標籤
    d_risk = row['Dynamic Risk']
    icon_color = risk_color_map.get(d_risk, 'blue') # 預設藍色以防萬一
    
    # 處理雨量數值顯示 (如果是 NaN 代表不在警戒範圍內)
    current_rain = row['rain_1hr'] if not pd.isna(row['rain_1hr']) else 0
    near_station = row['StationName']
    
    # 建立彈出視窗 HTML
    popup_html = f"""
    <div style="font-family: 'Microsoft JhengHei', sans-serif; width:220px;">
        <h4 style="margin-bottom: 5px;">{row['避難收容處所名稱']}</h4>
        <hr style="margin: 5px 0;">
        <b style="font-size: 1.1em;">動態風險：{d_risk}</b><br>
        <b>鄰近雨量站：</b> {near_station}<br>
        <b>即時雨量：</b> {current_rain:.1f} mm/hr<br>
        <b>地形風險：</b> {row['risk_level']}<br>
        <b>行政區：</b> {row['縣市及鄉鎮市區']}<br>
    </div>
    """
    #  <b>海拔高度：</b> {row.get('mean_elevation', 0):.1f} m
    
    # 建立標記
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup=folium.Popup(popup_html, max_width=300),
        icon=folium.Icon(color=icon_color, icon='home', prefix='fa' if icon_color=='darkred' else 'glyphicon'),
        tooltip=f"{row['避難收容處所名稱']} [{d_risk}]"
    ).add_to(shelter_layer)

# 4. 加入圖層到地圖
shelter_layer.add_to(m)


## 添加HeatMap 圖層

In [7]:
# 雨量強度分布
from folium.plugins import HeatMap

# 1. 建立包含 [緯度, 經度, 雨量權重] 的資料清單
# HeatMap 的第三個參數會被視為強度 (Intensity)
heat_data = [[row['lat'], row['lon'], row['rain_1hr']] for idx, row in gdf_rainfall.iterrows()]

# 2. 建立熱點圖層並加入地圖 m
# name='Rainfall Heatmap': 設定在圖層控制選單中顯示的名稱
# show=False: 預設為隱藏，讓使用者手動切換開啟
HeatMap(
    data=heat_data, 
    name='Rainfall Heatmap', 
    min_opacity=0.3,
    max=max(gdf_rainfall['rain_1hr']) if not gdf_rainfall.empty else 1,
    radius=25, 
    blur=15, 
    show=False
).add_to(m)

# 提示：此時地圖還不會顯示切換選單，我們將在 Cell [8] 加入 LayerControl
print(f"✅ 已成功建立包含 {len(heat_data)} 個點位的熱點圖層。")


✅ 已成功建立包含 1256 個點位的熱點圖層。


## 📝 AI 決策支援模組說明 (AI Advisory Implementation)

本模組成功將 **Gemini 3 Flash** 大型語言模型整合至 GIS 流程中，實現了從「數據監測」到「行動建議」的自動化轉化。

### 1. 安全與環境配置 (Security & Client Setup)
* **API 安全強化：** 導入 `os.getenv("api_key")` 機制。透過讀取系統環境變數，有效避免 API Key 在程式碼原始碼中暴露，符合開發安全規範。
* **模型選用：** 使用最新的 `gemini-3-flash-preview`。該模型具備極高的推理速度與精準的指令遵循能力，適合處理即時防災調度任務。

### 2. 專家系統 Prompt 工程 (Expert System Prompting)
* **角色設定：** 將 AI 定位為「花蓮與宜蘭地區防災指揮中心專家顧問」，賦予其在地化的專業語氣。
* **數據驅動：** 動態注入避難所名稱、行政區、地形風險、即時雨量及系統判定之動態風險等級。
* **輸出規範：** 強制要求「3 句話」精簡輸出，並包含「具體動作」（如撤離、物資調度），確保指揮官在緊急時刻能迅速判讀並執行。

### 3. 目標鎖定策略 (Targeted Analysis)
* **計算資源優化：** 透過 `sort_values` 篩選降雨量最高的前三名避難所（Top 3 Critical）。
* **策略性分析：** 避免對所有站點進行無差別呼叫，將 AI 運算能力集中在風險最高的「熱點區域」，大幅提升應變效率。

### 4. 空間視覺化整合 (Spatial Visual Integration)
* **獨立圖層管理：** 建立專屬的 `FeatureGroup`（🤖 AI 防災專家建議）。
* **UI/UX 優化：** * **Icon 標記：** 使用紅色 `person-rays` 圖示（FontAwesome），具備強烈的視覺警示作用。
    * **自定義 HTML：** 在 Popup 中設計了專屬的 CSS 樣式（包含紅色側邊線框），強化「指揮官建議」的權威感與易讀性。

---

In [8]:
from google import genai
import folium
import os

api_key = os.getenv("api_key")
# 1. 初始化 Client，使用新的 API Key，並考慮使用環境變數
client = genai.Client(api_key=api_key)

# 2. 定義 AI 建議函數
def get_ai_advice(row):
    prompt = f"""你是花蓮與宜蘭地區防災指揮中心的專家顧問。以下是即時監測數據：
    - 避難所名稱: {row['避難收容處所名稱']}
    - 行政區: {row['縣市及鄉鎮市區']}
    - 地形風險等級: {row['risk_level']}
    - 最近雨量站: {row['StationName']} (即時雨量: {row['rain_1hr']:.1f} mm/hr)
    - 系統判定動態風險: {row['Dynamic Risk']}

    請以防災專家身分，針對此避難所給出「指揮官緊急應變建議」。
    規範：請以 3 句話給出指揮官的緊急應變建議，語氣專業且精簡，包含具體動作（如：預防性撤離、加派物資、監測路徑）。"""
    
    try:
        # 使用新版 SDK 的請求方式
        response = client.models.generate_content(
            model='gemini-3-flash-preview', 
            contents=prompt
        )
        return response.text.strip()
    except Exception as e:
        # 印出錯誤細節方便除錯
        print(f"❌ AI 生成失敗 ({row['避難收容處所名稱']}): {e}")
        return "暫時無法取得 AI 建議，請依現行防災手冊應變。"

# ==========================================
# 3. 執行 AI 分析 (針對高風險前三名)
# ==========================================
# 假設 shelters_analysis 已經是準備好的 GeoDataFrame
top_3_critical = shelters_analysis.sort_values(by='rain_1hr', ascending=False).head(3)
ai_advice_dict = {}

print("--- 開始生成 AI 防災建議 ---")
for idx, row in top_3_critical.iterrows():
    print(f"正在分析: {row['避難收容處所名稱']}...")
    ai_advice_dict[row['避難收容處所名稱']] = get_ai_advice(row)

# ==========================================
# 4. 建立 Folium 圖層
# ==========================================
ai_layer = folium.FeatureGroup(name="🤖 AI 防災專家建議", show=True)

# 遍歷 top_3_critical 並標註
for idx, row in top_3_critical.to_crs(epsg=4326).iterrows():
    shelter_name = row['避難收容處所名稱']
    
    if shelter_name in ai_advice_dict:
        advice_text = ai_advice_dict[shelter_name]
        
        # 彈出視窗 HTML 樣式
        popup_content = f"""
        <div style="font-family: 'Microsoft JhengHei'; width: 250px;">
            <h4 style="color: #d9534f; margin-bottom: 5px;">🚨 指揮官應變建議</h4>
            <p style="margin-top: 0;"><b>地點：</b>{shelter_name}</p>
            <div style="background-color: #fff3f3; padding: 10px; border-left: 5px solid #d9534f; border-radius: 4px;">
                <span style="font-size: 0.95em; line-height: 1.5;">{advice_text}</span>
            </div>
        </div>
        """
        
        folium.Marker(
            location=[row.geometry.y, row.geometry.x],
            popup=folium.Popup(popup_content, max_width=300),
            icon=folium.Icon(color='red', icon='person-rays', prefix='fa'),
            tooltip=f"【AI 防災建議】{shelter_name}"
        ).add_to(ai_layer)

# 將圖層加入地圖 m (假設 m 已在地圖初始化部分定義)
ai_layer.add_to(m)

print("✅ AI 建議圖層已更新完成。")

--- 開始生成 AI 防災建議 ---
正在分析: 南安國中...
正在分析: 蘇東社區活動中心...
正在分析: 聖湖社區活動中心...
✅ AI 建議圖層已更新完成。


## 將互動式圖表儲存成HTML檔案，並展示

In [9]:
# 顯示圖表
from folium import LayerControl
# 1. & 2. 將圖層控制選單加入地圖 m
# collapsed=False 代表選單預設會展開，方便使用者直接切換
LayerControl(collapsed=False).add_to(m)

# 3. 儲存地圖
m.save("../output/ARIA_v3_Fungwong.html")

# 4. 在 Notebook 中顯示最終地圖
m